# Task Formulation and EDA


This notebook starts Team 10's final project for **Fundamentals of Natural Language / NLP-I** at Universitat Autonoma de Barcelona, academic year 2025-2026. Our team is Phoebe Iglesias (1713459), David Redrejo (1790336), and Pau Rossell (1750424). The project is supervised by Ernest Valveny and Lei Kang.


The competition is **UAB-ASHO AI Codification** (`uab-asho-ai-codification`). The objective is to predict exactly one ICD category prefix, `y_category`, for each clinical literal. The target is the first character of the full ICD `Code`, so the expected label space is 36 categories: digits `0`-`9` and letters `A`-`Z`.


We also use this notebook to connect the project with our follow-up with Lei Kang and the final presentation context. Before making claims about TF-IDF, SVMs, RoBERTa, or ensembles, we need to show that we understand the corpus, the annotations, the quality risks, and the limits of the data.
</details>


Guiding question: what exactly are we predicting, and why is it not trivial?


We begin by translating the assignment into a supervised learning problem.  The previous object is a clinical literal $x_i$ and the target is the first ICD character $y_i=\mathrm{firstchar}(Code_i)$.  Before choosing any model, we ask three questions:


Do the files really match the task definition?
Is the label distribution balanced enough for accuracy to be meaningful?
Are there repeated or ambiguous literals that could make a naive string-match solution misleading?


This matters because every later notebook depends on this diagnosis: if the data are imbalanced, we need macro-F1; if literals are ambiguous, we need validation and error analysis; if train and leaderboard look different, public Kaggle scores must be interpreted carefully.


This EDA is inspired by a **Data Engineering mindset**: before modeling, we inspect the data-generating process, data quality, distributions, missingness, duplicates, and possible leakage. Two team members are currently taking Data Engineering, while Phoebe already took it last year, so we wanted this notebook to reflect that discipline rather than jumping directly into a model.


This also connects to the NLP course material: the introductory slides discuss corpora, datasets, and linguistic data, and the Basic Text Processing part reminds us that choices such as lowercasing, accent removal, tokenization, and punctuation handling are not neutral. In clinical text, a slash, a digit, or an abbreviation can carry useful meaning.


No model is trained in this notebook.


We start by turning the assignment into a measurable classification problem.


We read the generated artefacts back into the notebook so each result is visible where it is discussed. The recurring score notation is:


$$\mathrm{Accuracy} = \frac{1}{N}\sum_i \mathbb{1}(\hat y_i = y_i)$$


$$\mathrm{MacroF1} = \frac{1}{K}\sum_{k=1}^K F1_k, \qquad \Delta_m = m_{candidate} - m_{baseline}$$


Accuracy tells us whether the model wins overall; macro-F1 tells us whether it is learning beyond the dominant ICD categories. The next notebook freezes the text decisions so every model sees the same input.


We first locate the project root and define shared folders. This matters because every later table, plot and submission must be written and read from a predictable place.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

def show_table(relative_path, title, rows=8):
    path = PROJECT_ROOT / relative_path
    print(f"\n{title} -> {relative_path}")
    if not path.exists():
        print('Missing artifact. Run the preparation cells above first.')
        return None
    df = pd.read_csv(path)
    display(df.head(rows))
    return df

def show_image(relative_path):
    path = PROJECT_ROOT / relative_path
    print(f"\nFigure -> {relative_path}")
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Missing figure. Run the preparation cells above first.')


Here we choose which tables answer the current chapter question. The selection is part of the narrative: each table should explain the next modelling decision.


In [ ]:
tables_to_show = [('reports/tables/data_schema_summary.csv', 'Data files and role'), ('reports/tables/category_distribution.csv', 'Target distribution'), ('reports/tables/eda_key_findings.csv', 'EDA conclusions')]
visible_tables = {}
for relative_path, title in tables_to_show:
    visible_tables[title] = show_table(relative_path, title)


This code performs the next concrete step in the chapter. We keep it visible so the result can be connected with the explanation that follows.


In [ ]:
# Compact interpretation panel built from the visible tables above.
for title, df in visible_tables.items():
    if df is None or df.empty:
        continue
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    print(f"\nInterpretation: {title}")
    if {'accuracy', 'macro_f1'}.issubset(df.columns):
        best = df.sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
        label = best.get('model_id', best.get('candidate_version', best.get('candidate', 'best row')))
        print(f"Best visible candidate: {label} | accuracy={best['accuracy']:.3f}, macro_f1={best['macro_f1']:.3f}")
    elif {'count', 'share'}.issubset(df.columns):
        top = df.sort_values('count', ascending=False).iloc[0]
        print(f"Dominant class/category: {top.iloc[0]} with share={top['share']:.3f}; this is why macro-F1 matters.")
    elif numeric_cols:
        print(df[numeric_cols].describe().round(3).loc[['mean','min','max']])
    else:
        print('This table is qualitative, so it anchors the modelling decision rather than a numeric score.')


Plots are shown immediately after the code that chooses them, so the visual pattern can be interpreted before moving to the next experiment.


In [ ]:
figures_to_show = ['reports/figures/fig_01_category_distribution.png', 'reports/figures/fig_05_train_vs_leaderboard_lengths.png']
for relative_path in figures_to_show:
    show_image(relative_path)


We first locate the project root and define shared folders. This matters because every later table, plot and submission must be written and read from a predictable place.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIGURES = PROJECT_ROOT / 'reports' / 'figures'
TABLES = PROJECT_ROOT / 'reports' / 'tables'
EDA = PROJECT_ROOT / 'outputs' / 'eda'


<details open><summary>Data Inventory and Annotation Contract</summary></details>


We first validate the CSV files and the target definition:


```python
y_category = Code.astype(str).str[0]
```


The validation command is reproducible and writes machine-readable artifacts.


This code performs the next concrete step in the chapter. We keep it visible so the result can be connected with the explanation that follows.


In [ ]:
import subprocess
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts' / 'analyze_data_annotations.py')], check=True)


This code performs the next concrete step in the chapter. We keep it visible so the result can be connected with the explanation that follows.


In [ ]:
schema_summary = pd.read_csv(TABLES / 'data_schema_summary.csv')
schema_summary


**Interpretation.** We expected one training file, one leaderboard file, and possibly an ICD dictionary. We found exactly that: `codification_data.csv`, `leaderboard_data.csv`, and `icd_d_p_pairs.csv`. The training schema is clean (`Code`, `Literal`), and the leaderboard file has `id`, `Literal`. It does not include `y_category`, which is normal for a Kaggle test file but important to record because the expected-column description included it.


The visual EDA command creates the figures and tables used below. Each plot answers a specific question rather than decorating the report.


This code performs the next concrete step in the chapter. We keep it visible so the result can be connected with the explanation that follows.


In [ ]:
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_visual_eda.py')], check=True)


**Question.** Which ICD category prefixes dominate the training data?


**What we expected.** Medical coding datasets are usually imbalanced, but because this task predicts broad prefixes rather than full ICD codes, we expected the imbalance to be less extreme than full-code prediction.


**What we found.** The label space has all 36 expected categories, but the distribution is still uneven. Category `Z` is the largest, followed by `O` and numeric procedure-like categories.


**How it affects the next step.** Accuracy alone will mostly reward common categories. Later evaluation must include per-category analysis and macro-style metrics even if the competition metric is strict accuracy.


This cell pulls saved artifacts back into the notebook. The point is to make the result visible where we discuss it, instead of asking the reader to trust files hidden in folders.


In [ ]:
display(Image(filename=FIGURES / 'fig_01_category_distribution.png'))
pd.read_csv(TABLES / 'category_distribution.csv').head(10)


<details open><summary>Long Tail of Full ICD Codes</summary></details>


**Question.** Does broad category prediction hide a harder long-tail problem underneath?


**What we expected.** ICD coding normally has many rare full codes, even when categories are broader.


**What we found.** The training set contains thousands of unique full ICD codes. Many appear very rarely, producing a long tail even though the final target has only 36 labels.


**How it affects the next step.** We should be careful when using full-code information for feature engineering or augmentation: rare codes may introduce noise, but they also explain why identical literals can map to different labels.


Plots are shown immediately after the code that chooses them, so the visual pattern can be interpreted before moving to the next experiment.


In [ ]:
display(Image(filename=FIGURES / 'fig_02_long_tail_distribution.png'))


<details open><summary>Literal Lengths and Short Context</summary></details>


**Question.** How much text does the model actually receive?


**What we expected.** Clinical literals are short, unlike long discharge summaries often used in ICD-coding papers.


**What we found.** The median training literal is only about two whitespace tokens. Tokenizer-level counts are left for the transformer phase because no tokenizer is loaded in this EDA.


**How it affects the next step.** This is closer to noisy terminology classification than long-document understanding. Character n-grams, abbreviation handling, and exact/near-exact matching are likely important.


Plots are shown immediately after the code that chooses them, so the visual pattern can be interpreted before moving to the next experiment.


In [ ]:
display(Image(filename=FIGURES / 'fig_03_literal_length_distribution.png'))


**Question.** Are some categories systematically described with longer or shorter literals?


**What we expected.** Procedure-like categories and pregnancy-related categories might have different phrase styles.


**What we found.** Most categories remain short, but the spread differs by category. Some categories have more variable descriptions, which may affect feature extraction.


**How it affects the next step.** We should avoid one-size-fits-all assumptions about literal length and inspect category-level errors later.


Plots are shown immediately after the code that chooses them, so the visual pattern can be interpreted before moving to the next experiment.


In [ ]:
display(Image(filename=FIGURES / 'fig_04_length_by_category.png'))


<details open><summary>Train vs Leaderboard Length Shift</summary></details>


**Question.** Does the leaderboard look like the training data based on the fields we can observe?


**What we expected.** If both files come from the same clinical-literal process, the length distributions should be close.


**What we found.** Train and leaderboard length distributions are very similar. About half of unique normalized leaderboard literals appear in training, which suggests both overlap and a meaningful unseen-literal challenge.


**How it affects the next step.** Exact matching will help but will not be enough. We need a model that generalizes from short noisy forms.


This cell pulls saved artifacts back into the notebook. The point is to make the result visible where we discuss it, instead of asking the reader to trust files hidden in folders.


In [ ]:
display(Image(filename=FIGURES / 'fig_05_train_vs_leaderboard_lengths.png'))
pd.read_csv(TABLES / 'train_leaderboard_shift_summary.csv')


**Question.** Which clinical shorthand patterns appear in the literals?


**What we expected.** We expected uppercase abbreviations, accents, digits, punctuation, and measurement-like strings.


**What we found.** Accents are common, around one in ten literals are all uppercase, and digits/punctuation appear often enough that removing them blindly could discard signal.


**How it affects the next step.** Lowercasing and accent removal can improve matching, but they can also collapse distinct forms. We should compare normalization choices rather than assume one is always best.


This cell pulls saved artifacts back into the notebook. The point is to make the result visible where we discuss it, instead of asking the reader to trust files hidden in folders.


In [ ]:
display(Image(filename=FIGURES / 'fig_06_text_pattern_presence.png'))
pd.read_csv(TABLES / 'text_pattern_summary.csv')


<details open><summary>Duplicates, Ambiguity, and Leakage Risk</summary></details>


**Question.** Do identical literals always mean the same code/category?


**What we expected.** Some duplication is expected, but clinical literals may be ambiguous because they are short and context-poor.


**What we found.** Many duplicate literals map to multiple full ICD codes, and many also map to multiple `y_category` labels. Examples such as pregnancy-related literals show that the same surface phrase can mean different things when the missing clinical context changes.


**How it affects the next step.** Random row-level splits may leak duplicate literals across train and validation. We should consider literal-level splits or at least report this risk clearly.


This code performs the next concrete step in the chapter. We keep it visible so the result can be connected with the explanation that follows.


In [ ]:
pd.read_csv(TABLES / 'duplicate_literal_analysis.csv').head(12)


<details open><summary>What This Data Lets Us Do / Does Not Let Us Do</summary></details>


**What it lets us do.** We can train supervised models for broad ICD category prediction, study class imbalance, use exact and near-exact literal matching, exploit character-level morphology, and compare train/leaderboard surface distributions.


**What it does not let us do.** It does not give patient context, surrounding notes, negation scope, chronology, or enough information to always disambiguate identical literals. It also does not directly evaluate full ICD-code prediction, because the official target is only the first character category.


**Risk for modeling.** A model may look strong by memorizing repeated literals or common categories while failing on rare or ambiguous cases. This is why the final report must include error analysis, not only a leaderboard/submission number.


<details open><summary>Understanding the Main Challenges of the Task</summary></details>


- **Class imbalance:** categories such as `Z` and `O` dominate, while rare categories have very few examples. The competition metric rewards exact category accuracy, so common classes can hide weak rare-class behavior.
- **Clinical ambiguity:** the same literal can map to several full codes and even several category prefixes. Some apparent model errors may reflect missing context.
- **Abbreviations and synonyms:** literals such as `HTA`, `VHC`, short Catalan/Spanish variants, and shorthand phrases require robust text processing.
- **Negation:** the literals are often too short to contain full negation context. If negation appears, it may be compressed or implicit.
- **Short literal context:** most examples contain only a few tokens, so long-document reasoning is not the main challenge.
- **Broad category vs full ICD code:** predicting the prefix is easier than predicting the full ICD code, but full-code diversity still shapes the data.
- **Competition metric:** strict `y_category` accuracy is simple and objective, but it should be complemented by per-class analysis during development.


This code performs the next concrete step in the chapter. We keep it visible so the result can be connected with the explanation that follows.


In [ ]:
pd.read_csv(TABLES / 'eda_key_findings.csv')


Chapter conclusion: what EDA tells the rest of the project


The EDA gives us the first real constraint of the project: the input texts are short, but the problem is not a toy problem.  We see imbalance, repeated literals, and surface patterns such as accents, uppercase abbreviations, digits and punctuation.  That means the next notebook should not rush into aggressive normalization.  Instead, preprocessing has to answer a careful question: which cleaning operations make the text easier to model, and which ones erase clinical signal?
